# yt-clips — Colab T4 Worker
## Runtime → T4 GPU (15GB VRAM)

Run all cells in order. The watcher listens for jobs pushed from your Mac
via `bridge.py` (tunnel → Drive API → local file fallback).

### Prerequisites
1. Create a **Colab Secret** named `OPENROUTER_API_KEY` with your key
2. Or upload `.env` to `yt-clips/` on Google Drive
3. **(Optional)** `client_secrets.json` + `yt_token.json` in Drive for auto-upload

In [ ]:
# CELL 1: Mount Drive + Load Environment + Pull Code
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os, sys, time, subprocess, threading, json
from pathlib import Path

# ── Locate project root on Drive ──
for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        os.chdir(p)
        print(f"Working dir: {p}")
        break
else:
    os.chdir("/content")
    print("yt-clips not found on Drive — running from /content")

# ── Load .env from Drive ──
if Path(".env").exists():
    for line in open(".env"):
        line = line.strip()
        if line and not line.startswith("#") and "=" in line:
            k, v = line.split("=", 1)
            os.environ[k.strip()] = v.strip()
    print(".env loaded from Drive")

# ── Colab secrets override ──
try:
    from google.colab import userdata
    for key in ["OPENROUTER_API_KEY", "GROQ_API_KEY", "GOOGLE_API_KEY",
                "OPENAI_API_KEY", "YOUTUBE_CLIENT_ID", "YOUTUBE_CLIENT_SECRET"]:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
            print(f"{key} loaded from secrets")
except Exception:
    print("No Colab secrets found (use .env instead)")

# ── Pull latest code ──
print("Pulling latest code...")
subprocess.run("git pull origin main 2>&1 | tail -5", shell=True)

# ── Verify automation module ──
sys.path.insert(0, ".")
from automation.env import is_colab, gpu_info
print(f"Colab env: {is_colab()}")

In [ ]:
# CELL 2: Install Dependencies (T4 GPU-optimised)
def run(cmd, timeout=180):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    if r.returncode != 0:
        err = (r.stderr.strip()[-200:] if r.stderr else r.stdout.strip()[-200:])
        print(f"  \u26a0 {cmd[:60]}... ({err})")
    return r

print("[1/4] System deps...")
run("apt-get update -qq && apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1")

print("[2/4] PyTorch + CUDA 12.1 (T4)...")
run("pip install -q torch torchvision torchaudio "
    "--extra-index-url https://download.pytorch.org/whl/cu121", timeout=300)

print("[3/4] Premium GPU packages (ultralytics, GFPGAN, Real-ESRGAN)...")
run("pip install -q ultralytics gfpgan basicsr realesrgan")

print("[4/4] Pipeline Python deps...")
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless "
    "numpy filterpy scipy google-genai google-generativeai openai "
    "python-dotenv Pillow requests "
    "google-api-python-client google-auth-httplib2 google-auth-oauthlib")

# ── torchvision compat (fixes basicsr/realesrgan/gfpgan) ──
try:
    import utils.torchvision_compat  # noqa: F401
    print("torchvision compat applied")
except Exception:
    pass

# ── Verify GPU ──
import torch
gpu_name = gpu_info().get("name", "unknown")
cuda_ok = torch.cuda.is_available()
print(f"\nPyTorch CUDA: {cuda_ok}  |  GPU: {gpu_name}")
if cuda_ok:
    print(f"VRAM: {gpu_info()['memory_total_gb']}GB")
else:
    print("\u26a0 CUDA unavailable — set Runtime > Change runtime type > T4 GPU")

In [ ]:
# CELL 3: Start Watcher + TunnelKeeper
PORT = int(os.environ.get("PORT", "5000"))

# ── Kill stale processes ──
for pat in ["python watcher.py", "ssh serveo", "ssh localhost.run", "localtunnel"]:
    subprocess.run(f"pkill -f '{pat}' 2>/dev/null || true", shell=True)
subprocess.run(f"fuser -k {PORT}/tcp 2>/dev/null || true", shell=True)
time.sleep(2)

# ── Create data dirs ──
for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs", "photos"]:
    Path(folder).mkdir(exist_ok=True)

# ── Start Watcher (subprocess → watcher.log) ──
watcher_proc = subprocess.Popen(
    [sys.executable, "watcher.py"],
    stdout=open("watcher.log", "w"), stderr=subprocess.STDOUT,
)
time.sleep(3)

import urllib.request
for _ in range(10):
    try:
        r = urllib.request.urlopen(f"http://localhost:{PORT}/health", timeout=3)
        health = json.loads(r.read())
        print(f"\u2705 Watcher: http://0.0.0.0:{PORT}  (processing={health['processing']})")
        break
    except Exception:
        time.sleep(2)
else:
    print(f"\u274c Watcher failed — log:")
    if Path("watcher.log").exists():
        for l in open("watcher.log").read().strip().splitlines()[-5:]:
            print(f"  {l}")

# ── Start TunnelKeeper (auto-reconnect, 3 fallback methods) ──
from automation.tunnel import TunnelKeeper

keeper = TunnelKeeper(port=PORT, url_file=Path("colab_url.txt"))
tunnel_url = keeper.start()

if tunnel_url:
    print(f"\u2705 Tunnel: {tunnel_url}")
else:
    print("\u26a0 Tunnel connecting in background...")
    print("  (serveo.net \u2192 localhost.run \u2192 localtunnel)")
    print("  Jobs fall back to Drive API / file poll.")

# ── Enable premium features for T4 ──
import yaml
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)
if not cfg.get("premium", {}).get("enabled"):
    cfg["premium"] = {"enabled": True, "face_enhancement": True,
                      "frame_interpolation": True, "host_ref_photos": ""}
    cfg["enhancement"]["ref_grade"] = True
    cfg["export"]["super_resolution"] = True
    with open("config.yaml", "w") as f:
        yaml.dump(cfg, f, default_flow_style=False)
    print("\u2705 Premium features enabled (T4 GPU)")
else:
    print("Premium already enabled")

In [ ]:
# CELL 4: Dashboard (keep running)
from automation.tunnel import tunnel_status
from automation.memory import memory_report
from automation.env import gpu_info

ts = tunnel_status()
print("=" * 55)
print("  WORKER IS ONLINE")
print("=" * 55)
print(f"  Tunnel: {ts.get('url') or 'connecting...'}")
print()
print("  On your Mac:")
print('  python bridge.py "https://youtu.be/VIDEO_ID"')
print()
print("  Or run pipeline directly here:")
print('  !python pipeline.py "https://youtu.be/VIDEO_ID"')
print()

try:
    last_pos = 0
    while True:
        time.sleep(12)

        log_path = Path("watcher.log")
        if log_path.exists():
            with open(log_path) as f:
                f.seek(last_pos)
                for l in f:
                    if l.strip():
                        print(l.strip())
                last_pos = f.tell()

        mem = memory_report()
        gpu = gpu_info()
        ts = tunnel_status()
        print(f"[MEM {mem.get('free_gb', 0):.1f}GB free  "
              f"GPU {gpu.get('name', '?')[:20]} {gpu.get('memory_free_gb', 0):.1f}GB  "
              f"Tunnel {'OK' if ts.get('alive') else '...'}  "
              f"uptime {ts.get('uptime', 0):.0f}s]")
except KeyboardInterrupt:
    print("\nStopped.")